# MAGI low-statistics ladder --- CR

**The demonstration the tool rests on, and which has never been run.** MAGI's premise is
that it is for the regime where full Monte Carlo will not converge. That is asserted in
the paper, not shown. This trains the *same architecture* on progressively smaller
subsamples of the corrected CR training set and asks where it breaks.

Two axes, from one set of runs:

1. **Fidelity** --- does the generated population still match the truth?
2. **Memorisation** --- at small N a generative model can score well by *recalling*.
   The nearest-neighbour harness (validated 17--18/08: held-out real reads 1.00,
   literal copies read 0.00) is run at every rung. A model trained on 10^3 crossings
   that still generalises is the argument; the same model caught memorising is the
   refutation. **Same figure, both answers.**

## Design choices, so the ladder measures data and not recipe

- **Architecture is fixed**, rebuilt at every rung from the full-size checkpoint's own
  `mix_CR_config.json`. Identical latent dim, hidden sizes, flow bins, warp knots, prior
  and line table. Only the training data changes.
- **Line positions are inherited** from the full run rather than re-detected. At 10^3
  crossings line detection is not meaningful, and the line table is prior physics
  knowledge (EADL), not something a small sample should have to rediscover.
- **Batch size scales with N** (a 4096 batch on 700 training rows is one step per epoch),
  and the epoch cap rises for small rungs. Early stopping on `val_loss` decides where to
  stop, so each rung trains to convergence on whatever data it has.
- **Transformers are refit per rung.** A real low-statistics user has only their N
  crossings; fitting the quantile transforms on the full set would leak.

Everything is saved to Drive. The fidelity and memorisation analysis runs **locally**,
against the harness that has already been validated in both directions.

## 0 · Colab runtime setup

In [ ]:
# Confirm we actually have a GPU before spending an hour.
import subprocess
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv"],
                     capture_output=True, text=True).stdout or "NO GPU -- Runtime > Change runtime type > GPU")


In [ ]:
# MOUNT FIRST. Everything below needs Drive to exist.
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE = "/content/drive/MyDrive/MAGI"
TRAINING_DATA_DIR = f"{DRIVE}/TrainingData"
CR_FILE = f"{TRAINING_DATA_DIR}/alloutputDSCryoSphereCR_ingoingfix.dat"

need = [DRIVE, f"{DRIVE}/MAGI_package", f"{DRIVE}/MAGI_package/setup.py",
        TRAINING_DATA_DIR, CR_FILE, f"{DRIVE}/CandidateLines"]
missing = [p for p in need if not os.path.exists(p)]
for p in need:
    print(("OK   " if os.path.exists(p) else "MISS "), p)
assert not missing, f"upload these to Drive first: {missing}"

with open(CR_FILE) as f:
    first = f.readline().split()
print(f"\ncolumns = {len(first)} (expect 13)")
print("first row:", " ".join(first[:7]), "...")


In [ ]:
# ============================================================================
# Install magi.  Two sources; pick one.
#
#   "github" - clone the repo with a token from Colab Secrets. The token is
#              read from the secret store at runtime, never typed into a cell
#              and never written into this notebook. Gives a PINNED commit,
#              recorded below and saved into the checkpoint metadata.
#   "drive"  - install the copy you synced to Drive. No token needed.
#
# Either way TrainingData/ is gitignored, so the 339 MB .dat still comes from
# Drive. GitHub only supplies MAGI_package/ and CandidateLines/.
# ============================================================================
PACKAGE_SOURCE = "github"       # "github" or "drive"
GIT_REF        = "energy-mixture"

import os, sys, shutil, subprocess

REPO_LOCAL = "/content/MAGI_repo"


def run(cmd, secret=None):
    """Run a command, scrubbing `secret` out of everything printed.
    pip and git both echo the URL they were given -- without this the token
    would land in saved notebook output."""
    r = subprocess.run(cmd, capture_output=True, text=True)
    clean = lambda s: s.replace(secret, "***") if secret else s
    if r.returncode != 0:
        print("FAILED:", clean(" ".join(cmd)))
        print(clean(r.stderr)[-3000:])
    return r


if PACKAGE_SOURCE == "github":
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")     # Secrets panel, key icon, left sidebar
    assert TOKEN, "no GITHUB_TOKEN secret -- add it in the Secrets panel"

    url = f"https://{TOKEN}@github.com/francescomonastra/MAGI.git"
    if os.path.exists(REPO_LOCAL):
        shutil.rmtree(REPO_LOCAL)
    # blob:none + sparse: fetch only the two directories we need, not
    # trained_models/ (189 files) or Plots/ (80).
    run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
         "--branch", GIT_REF, url, REPO_LOCAL], secret=TOKEN)
    run(["git", "-C", REPO_LOCAL, "sparse-checkout", "set",
         "MAGI_package", "CandidateLines"], secret=TOKEN)
    del TOKEN, url                            # do not leave it in the kernel

    PKG_DIR   = f"{REPO_LOCAL}/MAGI_package"
    LINES_DIR = f"{REPO_LOCAL}/CandidateLines"
    GIT_SHA = subprocess.run(["git", "-C", REPO_LOCAL, "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip()
    print(f"cloned {GIT_REF} @ {GIT_SHA[:12]}")
else:
    PKG_DIR   = "/content/MAGI_package"
    LINES_DIR = f"{DRIVE}/CandidateLines"
    if os.path.exists(PKG_DIR):
        shutil.rmtree(PKG_DIR)
    shutil.copytree(f"{DRIVE}/MAGI_package", PKG_DIR)   # local disk: Drive FUSE
    GIT_SHA = "drive-copy-unpinned"                     # breaks provenance on purpose
    print("installed from the Drive copy -- provenance is whatever was synced")

r = run([sys.executable, "-m", "pip", "install", PKG_DIR])
if r.returncode != 0:
    print("falling back to sys.path (magi is pure Python)")
    sys.path.insert(0, PKG_DIR)

r2 = run([sys.executable, "-m", "pip", "install", "tensorflow_probability[tf]"])
print("tfp:", "ok" if r2.returncode == 0 else "FAILED (see above)")
print("package :", PKG_DIR)
print("lines   :", LINES_DIR)
print("provenance:", GIT_SHA)


In [ ]:
# Import check. If this fails, stop here -- nothing below can work.
import importlib, sys
importlib.invalidate_caches()
for m in [k for k in list(sys.modules) if k == "magi" or k.startswith("magi.")]:
    del sys.modules[m]

import magi
print("magi   :", magi.__file__)
print("version:", magi.__version__)

import tensorflow as tf, tensorflow_probability as tfp
print("tf     :", tf.__version__, "| tfp:", tfp.__version__)
print("GPUs   :", tf.config.list_physical_devices("GPU"))


In [ ]:
# ---------------------------------------------------------------- ladder config
import json, os, time, numpy as np

DRIVE   = "/content/drive/MyDrive/MAGI_data"
FULL    = f"{DRIVE}/TrainingData/alloutputDSCryoSphereCR_ingoingfix.dat"
REF_CKPT= f"{DRIVE}/trained_models/v0_8_2_CR_ingoingfix"
OUT     = f"{DRIVE}/ladder_CR"
os.makedirs(OUT, exist_ok=True)

RUNGS = [1_000, 3_000, 10_000, 30_000, 100_000, 300_000, 1_000_000, None]  # None = full
SEED  = 42
GEN_PER_RUNG = 200_000        # generated sample kept per rung, for the local analysis

for p in (FULL, f"{REF_CKPT}/mix_CR_config.json"):
    print(("OK   " if os.path.exists(p) else "MISS "), p)

REF_CONFIG = json.load(open(f"{REF_CKPT}/mix_CR_config.json"))
print("\nreference architecture:", REF_CONFIG["model_class"])
print("  latent", REF_CONFIG["latent_dim"], "| hidden", REF_CONFIG["hidden"],
      "| flow bins", REF_CONFIG["continuum_flow_bins"],
      "| lines", len(REF_CONFIG["line_positions_y"]))


In [ ]:
# ---------------------------------------------------------------- load once
import magi, pandas as pd

center = (0.0, 0.0, -507.66)
df_full = magi.load_detector_table(filepath=FULL, sep=r"\s+")

# Radius measured from the data, never assumed (the R=100 fallback cost us a
# ~1.7% aimed-flux bias before it was caught on 18/08).
_p = df_full[["X", "Y", "Z"]].to_numpy(float)
R = float(np.median(np.linalg.norm(_p - np.asarray(center), axis=1)))
print(f"{len(df_full):,} crossings | measured R = {R:.4f} mm")
del _p


In [ ]:
# ---------------------------------------------------------------- one rung
X_IFU_RESOLUTION_EV = 4.0
FWHM_MEV = X_IFU_RESOLUTION_EV * 1e-6

# Line table INHERITED from the full-size run (prior physics, EADL). At 10^3
# crossings line detection is not meaningful, so re-detecting per rung would
# measure the detector, not the model.
LINE_Y   = np.asarray(REF_CONFIG["line_positions_y"], dtype=np.float32)
LINE_MEV = (10.0 ** LINE_Y).astype(np.float64)
MATCHED  = [{"label": f"line_{i}", "candidate_energy_mev": float(e)}
            for i, e in enumerate(LINE_MEV)]
print(f"inherited {len(MATCHED)} line(s) at {np.round(LINE_MEV*1e3, 3)} keV")


def run_rung(n_sub, seed=SEED):
    """Train the reference architecture on a subsample of n_sub crossings."""
    t0 = time.time()
    tag = "full" if n_sub is None else f"n{n_sub}"
    df = (df_full if n_sub is None
          else df_full.sample(n=n_sub, random_state=seed).reset_index(drop=True))

    prep = magi.build_physical_features(df, center=center, radius=R)
    feature_pack = magi.build_feature_dataframe(
        prep, energy_binning_mode="log_fixed_count", n_bins=512,
        geometry_transform="quantile_u_r_u_v_phi_r_phi_v",
        n_quantiles=min(10000, max(10, len(df) // 2)),
        random_state=seed, energy_transform="log10")

    # gate targets -> the extra continuous columns the v0.8 head consumes
    E_full = feature_pack["filtered_prep"]["features"]["Energy"].to_numpy()
    gate_targets = magi.build_gate_targets(
        E_full, feature_pack["energy_bins"], MATCHED,
        bandwidth_mode="resolution", bandwidth_fwhm_mev=FWHM_MEV)
    feat = feature_pack["feat"].copy()
    for j in range(gate_targets.shape[1]):
        feat[f"gate_target_{j}"] = gate_targets[:, j]
    cont_cols = ("u_r_q", "u_v_q", "phi_r_q", "phi_v_q", "energy_y") + tuple(
        f"gate_target_{j}" for j in range(gate_targets.shape[1]))

    # A species needs enough members to survive a stratified 70/15/15 split.
    # e+ is 0.12% of CR, so at n=1000 it has ONE member and sklearn refuses.
    # Requiring ~20 members is the honest low-statistics behaviour: a species
    # you have fewer than 20 examples of cannot be learned, and n_types falling
    # at small N is a RESULT to report, not a bug to work around.
    prob_threshold = max(1e-5, 20.0 / len(df))
    dataset_pack = magi.filter_particle_types_continuous_geometry(
        feat=feat, prob_threshold=prob_threshold, cont_cols=cont_cols)
    n_types     = dataset_pack["n_types"]
    idx_to_type = dataset_pack["idx_to_type"]
    type_probs  = dataset_pack["type_probs"]

    # zone_probs are EMPIRICAL, so they are recomputed per rung -- unlike the
    # architecture, which is fixed. Losing a zone at small N is a real result.
    n_zones = gate_targets.shape[1]
    zc = dataset_pack["X_cont_raw"][:, -n_zones:]
    yt = dataset_pack["y_type"]
    zone_probs = np.zeros((n_types, n_zones))
    for t in range(n_types):
        m = (yt == t)
        row = zc[m].mean(axis=0) if m.any() else np.zeros(n_zones)
        zone_probs[t] = row / row.sum() if row.sum() > 0 else np.eye(n_zones)[0]

    split_pack  = magi.split_feature_data(dataset_pack, test_size_total=0.30,
                                          val_size_from_temp=0.50, random_state=seed)
    scaled_pack = magi.scale_continuous_features(split_pack, scale_cols=())
    cond_pack   = magi.build_conditioning_and_weights(
        scaled_pack, n_types=n_types, idx_to_type=idx_to_type, alpha=0.5)

    n_train = len(split_pack["idx_train"])
    batch   = int(min(4096, max(32, 2 ** int(np.floor(np.log2(max(8, n_train / 8)))))))
    tf_pack = magi.build_tf_datasets(cond_pack, batch_size=batch,
                                     shuffle_buffer_cap=200_000)
    steps   = max(1, n_train // batch)
    epochs  = int(min(400, max(40, 25_000 // steps)))

    cfg = REF_CONFIG
    model = magi.CVAE_MixEnergy_ContPhi_TaskAdaptive(
        n_types=n_types, line_positions_y=LINE_Y,
        latent_dim=cfg["latent_dim"], hidden=tuple(cfg["hidden"]), beta=cfg["beta"],
        continuum_mode=cfg["continuum_mode"],
        continuum_flow_bins=cfg["continuum_flow_bins"],
        continuum_flow_transforms=cfg["continuum_flow_transforms"],
        continuum_flow_warp=cfg["continuum_flow_warp"],
        continuum_flow_warp_y_knots=np.asarray(cfg["continuum_flow_warp_y_knots"]),
        continuum_flow_warp_z_knots=np.asarray(cfg["continuum_flow_warp_z_knots"]),
        energy_flow_condition=cfg["energy_flow_condition"], prior=cfg["prior"],
        # w_gate_aux and line_logsigma_init are TRAINING-time settings and are
        # absent from to_generation_config(); take them from the v0.8.2 recipe.
        w_gate_aux=2.0, gate_focal_gamma=cfg["gate_focal_gamma"],
        gate_class_weights=cfg.get("gate_class_weights"),
        line_logsigma_init=magi.line_logsigma_from_resolution(
            LINE_MEV, X_IFU_RESOLUTION_EV, fwhm=True),
        line_logsigma_trainable=False,
        prior_zone_conditioning=cfg["prior_zone_conditioning"],
        zone_probs=zone_probs)
    magi.compile_model(model, learning_rate=2e-4)
    cbs = magi.build_default_callbacks()          # takes no model argument

    hist = magi.fit_model(model=model, train_ds=tf_pack["train_ds"],
                          val_ds=tf_pack["val_ds"], epochs=epochs,
                          callbacks=cbs, verbose=0)

    save_dir = f"{OUT}/{tag}"
    os.makedirs(save_dir, exist_ok=True)
    magi.save_final_trained_model(
        model=model, save_dir=save_dir, model_name=f"mix_CR_{tag}", history=hist,
        model_config=model.to_generation_config(),
        preprocessing_metadata={
            "source": "CR", "center": list(center), "radius": R,
            "geometry_transform": "quantile_u_r_u_v_phi_r_phi_v",
            "energy_transform": "log10", "n_types": n_types,
            "idx_to_type": idx_to_type, "type_probs": list(np.asarray(type_probs)),
            "cont_cols": list(cont_cols),
            "energy_bins": list(np.asarray(feature_pack["energy_bins"]).ravel()),
            "geometry_metadata": feature_pack.get("geometry_metadata"),
            "ladder_n": n_sub, "n_train": n_train},
        training_metadata={"ladder_n": n_sub, "n_train": n_train, "batch": batch,
                           "epochs_cap": epochs, "epochs_run": len(hist.history["loss"]),
                           "steps_per_epoch": steps, "seed": seed,
                           "device": "colab-gpu", "wall_s": round(time.time() - t0)},
        callbacks=cbs,
        notes=f"Low-statistics ladder rung {tag}. Architecture from the full-size "
              f"mix_CR_config.json; lines inherited; zone_probs per rung; only N differs.")
    import joblib
    joblib.dump(feature_pack["quantile_transformers"],
                f"{save_dir}/mix_CR_{tag}_quantile_transformers.joblib")

    rec = {"tag": tag, "n_sub": n_sub, "n_train": n_train, "n_types": n_types,
           "types": list(idx_to_type.values()) if hasattr(idx_to_type, "values") else None,
           "batch": batch, "epochs_run": len(hist.history["loss"]),
           "val_loss": float(hist.history["val_loss"][-1]),
           "wall_s": round(time.time() - t0)}
    print(f"  {tag:>10s} n_train={n_train:>9,} types={n_types} batch={batch:>5d} "
          f"epochs={rec['epochs_run']:>3d} val_loss={rec['val_loss']:+.3f} {rec['wall_s']:>5d}s")
    return rec


In [ ]:
# ---------------------------------------------------------------- run the ladder
results = []
for n in RUNGS:
    try:
        results.append(run_rung(n))
    except Exception as e:
        import traceback
        print(f"  rung {n}: FAILED -- {type(e).__name__}: {e}")
        traceback.print_exc()          # full trace, so a failure is diagnosable
        results.append({"tag": str(n), "n_sub": n,
                        "error": f"{type(e).__name__}: {e}"})
    with open(f"{OUT}/ladder_summary.json", "w") as f:
        json.dump(results, f, indent=2, default=str)

print("\n=== ladder complete ===")
for r in results:
    print(r)

# Generation is deliberately NOT done here. scripts/generate_geant_source.py is a
# tested path that has produced four correct samples this week, and it enforces
# the sphere gate. Download the checkpoints and generate locally.


## Bring back

The whole `MyDrive/MAGI_data/ladder_CR/` directory. Each rung holds its checkpoint,
its **own** quantile transformers (keep them together --- separating them produces
silently wrong physics), and a 200k generated sample.

Locally:

```bash
python tools/memorisation_test.py cr        # per rung, once the samples are in place
```

The expected shape of the result: fidelity degrades smoothly as N falls, while the
memorisation ratio drops toward the memoriser curve at the smallest rungs. **Where those
two cross is the tool's honest lower limit**, and that number is the paper's claim about
the low-statistics regime.

If a small rung *fails to train* rather than degrading, that is also a result --- record
it, do not tune it away. Note too that `n_types` may fall below 4 at the smallest rungs:
e+ is 0.12% of the population, so 10^3 crossings hold roughly one. That is a real
low-statistics failure mode, not a bug.